In [1]:
from google.colab import files
import torch
import torchaudio
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText

/usr/local/lib/python3.10/dist-packages/transformers/deepspeed.py:24: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


In [2]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# CPUs generally work best with "float32" or "bfloat16".
# "float16" is primarily optimized for GPU usage.
TORCH_DTYPE = torch.float16 if torch.cuda.is_available else torch.float32

# Load audio file

Click `Browse`and load an audio file

In [3]:
uploaded = files.upload()

Saving data_fairy_tail_01.mp3 to data_fairy_tail_01 (2).mp3


In [4]:
audio_path = list(uploaded.keys())[0]

# Seamless m4t V2 Large (facebook)

In [5]:
MODEL_ID = "facebook/seamless-m4t-v2-large"

In [6]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = SeamlessM4Tv2ForSpeechToText.from_pretrained(MODEL_ID)
model = model.to(DEVICE)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
audio, orig_freq =  torchaudio.load(audio_path)
audio =  torchaudio.functional.resample(audio, orig_freq=orig_freq, new_freq=16_000) # must be a 16 kHz waveform array
audio_inputs = processor(
    audios=audio,
    src_lang="kat",
    return_tensors="pt",
    sampling_rate=16_000
)
audio_inputs.to(DEVICE)

{'input_features': tensor([[[-8.7002, -8.4128, -9.3063,  ..., -6.7629, -6.7946, -6.6874],
         [-7.8117, -7.3156, -8.0154,  ..., -1.7805, -1.5645, -1.7415],
         [ 0.6095,  0.4081,  0.2073,  ...,  1.9277,  1.8626,  1.7575],
         ...,
         [-5.2165, -5.0309, -5.6145,  ..., -2.8170, -2.8296, -2.9469],
         [-7.4184, -6.9479, -8.0027,  ..., -3.6295, -3.7997, -3.4971],
         [-8.7002, -8.4128, -9.3063,  ...,  0.0000,  0.0000,  0.0000]]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 0]], device='cuda:0', dtype=torch.int32)}

In [8]:
decoder_input_ids = model.generate(
    **audio_inputs,
    tgt_lang="kat",
    # task="ASR",
    #generate_speech=False
)

OutOfMemoryError: CUDA out of memory. Tried to allocate 6.57 GiB. GPU 0 has a total capacity of 14.75 GiB of which 1.60 GiB is free. Process 473195 has 13.15 GiB memory in use. Of the allocated memory 12.89 GiB is allocated by PyTorch, and 140.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [24]:
decoder_input_ids

tensor([[     3, 256044,   3543,  31529,  13037,   4837,  27080,   1889, 149572,
          24558,   5522, 169515,   2157, 212019,  42123,  45767,  15344, 126073,
          40608, 127250, 247676,      3]], device='cuda:0')

In [26]:
translated_text = processor.decode(
    decoder_input_ids[0].tolist(),
    skip_special_tokens=True
)

In [28]:
print(translated_text)

ფინიკიური ქალაქების სანაპიროებზე განლაგება ვაჭრობის განვითარებას უწყობდა ხელს.
